In [1]:
import requests
import pandas as pd

url = "https://dadosabertos.camara.leg.br/api/v2/deputados"
headers = {"Accept": "application/json"}
params = {
    "itens": 100,
    "ordem": "ASC",
    "ordenarPor": "nome"
}

todos_deputados = []

with requests.Session() as session:
    while url:
        response = session.get(url, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        
        todos_deputados.extend(payload.get("dados", []))
        
        # Busca o link da próxima página; se não existir, encerra o loop
        links = payload.get("links", [])
        url = next((link["href"] for link in links if link.get("rel") == "next"), None)
        
        # Os parâmetros já vêm embutidos na URL do link 'next'
        params = None

df_deputados = pd.DataFrame(todos_deputados)
#display(df_deputados.head(10))

In [2]:
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

# 1. Garante que a raiz do projeto esteja no sys.path e importa as credenciais do main.py
root_path = Path.cwd().resolve().parents[1]
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from main import db_host, db_name, db_password, db_port, db_schema, db_user

# 2. Prepara o DataFrame para a camada Bronze (renomeia colunas e adiciona timestamp de carga)
df_bronze = df_deputados.rename(columns={
    "id": "n_id_deputado",
    "uri": "s_ds_uri",
    "nome": "s_ds_nome",
    "siglaPartido": "s_cd_sigla_partido",
    "uriPartido": "s_ds_uri_partido",
    "siglaUf": "s_cd_sigla_uf",
    "idLegislatura": "n_id_legislatura",
    "urlFoto": "s_ds_url_foto",
    "email": "s_ds_email"
}).copy()

# Trata possíveis valores vazios/NaN como None (vira NULL no PostgreSQL)
df_bronze = df_bronze.where(pd.notnull(df_bronze), None)
df_bronze["d_dh_carga"] = datetime.now()

colunas_bronze = [
    "n_id_deputado",
    "s_ds_uri",
    "s_ds_nome",
    "s_cd_sigla_partido",
    "s_ds_uri_partido",
    "s_cd_sigla_uf",
    "n_id_legislatura",
    "s_ds_url_foto",
    "s_ds_email",
    "d_dh_carga"
]
df_bronze = df_bronze[colunas_bronze]

# 3. DDL da tabela bronze_deputados (recria do zero) e comando de inserção
ddl_bronze = f"""
DROP TABLE IF EXISTS {db_schema}.bronze_deputados;

CREATE TABLE {db_schema}.bronze_deputados (
    n_id_deputado       INTEGER PRIMARY KEY,
    s_ds_uri            TEXT,
    s_ds_nome           VARCHAR(150),
    s_cd_sigla_partido  VARCHAR(20),
    s_ds_uri_partido    TEXT,
    s_cd_sigla_uf       CHAR(2),
    n_id_legislatura    INTEGER,
    s_ds_url_foto       TEXT,
    s_ds_email          VARCHAR(150),
    d_dh_carga          TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

insert_bronze = f"""
INSERT INTO {db_schema}.bronze_deputados (
    n_id_deputado, s_ds_uri, s_ds_nome, s_cd_sigla_partido,
    s_ds_uri_partido, s_cd_sigla_uf, n_id_legislatura,
    s_ds_url_foto, s_ds_email, d_dh_carga
) VALUES %s
ON CONFLICT (n_id_deputado) DO UPDATE SET
    s_ds_uri           = EXCLUDED.s_ds_uri,
    s_ds_nome          = EXCLUDED.s_ds_nome,
    s_cd_sigla_partido = EXCLUDED.s_cd_sigla_partido,
    s_ds_uri_partido   = EXCLUDED.s_ds_uri_partido,
    s_cd_sigla_uf      = EXCLUDED.s_cd_sigla_uf,
    n_id_legislatura   = EXCLUDED.n_id_legislatura,
    s_ds_url_foto      = EXCLUDED.s_ds_url_foto,
    s_ds_email         = EXCLUDED.s_ds_email,
    d_dh_carga         = EXCLUDED.d_dh_carga;
"""

# Converte os registros para tuplas nativas do Python
registros = [tuple(x) for x in df_bronze.itertuples(index=False, name=None)]

# 4. Executa no PostgreSQL (Neon)
with psycopg2.connect(
    host=db_host,
    dbname=db_name,
    user=db_user,
    password=db_password,
    port=db_port,
    sslmode="require"
) as conn:
    with conn.cursor() as cur:
        cur.execute(ddl_bronze)
        execute_values(cur, insert_bronze, registros)
    conn.commit()

print(f"Sucesso! Tabela recriada e {len(registros)} registros gravados em '{db_schema}.bronze_deputados'.")

Sucesso! Tabela recriada e 513 registros gravados em 'data_challenges_fase_1.bronze_deputados'.


In [3]:
# with psycopg2.connect(
#     host=db_host, dbname=db_name, user=db_user,
#     password=db_password, port=db_port, sslmode="require"
# ) as conn:
#     df_validacao = pd.read_sql(f"SELECT * FROM {db_schema}.bronze_deputados LIMIT 10;", conn)

# display(df_validacao)